# GPT-2 generation under encryption — the C++ driver

A prompt in, a continuation out, under CKKS. The prompt is prefilled, the model hands off
to the decode phase and then generates token by token with **encrypted argmax feedback**:
the sampled token never leaves the ciphertext domain until the end.

The run is **eager**: the checkout ships no plan for the C++ driver (the shipped plans are the
Python implementation's), and eager generation refreshes reactively whenever a ciphertext runs
out of levels.

Only the embedding (`wte` + `wpe`) runs client-side. The environment comes from the runner
(`NB=gpt2_torch bash scripts/run_notebooks.sh`).

<!-- prereq-cell -->
> **Prerequisites.** This notebook loads artifacts it does not create: the exported
> weights (`weights.bin.zip`, `client.npz`), the calibrated `configs.json`, and the decode
> oracle (`ALL_BLOCKS_IO_DIR`), which every run of the driver requires even when, as here,
> it is not read. Run **`setup_artifacts.ipynb`** first if you do not have them.

In [ ]:
import os
import time

import torch                                   # torch before _core (NCCL load order)
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from perseus import _core

# ── write your prompt here ────────────────────────────────────────────────────
PROMPT = ("The history of computing begins long before the first electronic machine, "
          "with mechanical devices built to automate arithmetic, the patient work of "
          "human calculators, and the slow realisation that calculation itself could "
          "be described precisely enough for a machine to carry it out.")
# ──────────────────────────────────────────────────────────────────────────────
P = int(os.environ.get("GEN_PROMPT", "8"))     # prompt tokens (the prefilled prefix)
N = int(os.environ.get("GEN_TOKENS", "10"))    # tokens to generate

tok = GPT2TokenizerFast.from_pretrained("gpt2")
hf = GPT2LMHeadModel.from_pretrained("gpt2").eval()

ids = tok(PROMPT)["input_ids"]
if len(ids) < P:
    raise ValueError(f"PROMPT is {len(ids)} tokens, need >= GEN_PROMPT={P}")
ids = ids[:P]
print(f"prompt ({P} tokens): {tok.decode(ids)!r}")

## Client side: embed the prompt

`wte[token] + wpe[position]` — the block-0 input, which is what the encrypted run
consumes. `run_generate` takes these rows directly, so the prompt below is the one the
model actually continues.

In [ ]:
with torch.no_grad():
    emb = (hf.transformer.wte(torch.tensor(ids)) +
           hf.transformer.wpe(torch.arange(P))).numpy()

rows = emb.tolist()                                 # generation feeds back its own argmax
print(f"embedded {P} tokens x {emb.shape[1]}")

# plaintext reference: greedy continuation from the same prompt
with torch.no_grad():
    ref = hf.generate(torch.tensor([ids]), max_new_tokens=N, do_sample=False,
                      pad_token_id=tok.eos_token_id)[0, P:].tolist()
print(f"plaintext continuation: {tok.decode(ref)!r}")

## Encrypted prefill → hand-off → generate

`RunConfig.from_env()` reads the model paths and the CKKS environment; `plan_dir` is
cleared explicitly so the run is eager whatever the shell exported. The prompt rows go in
directly. `res.bootstraps` is the number of reactive refreshes the run executed.

In [ ]:
cfg = _core.RunConfig.from_env()
cfg.plan_dir = ""                                   # eager: no plan
cfg.gen_prompt, cfg.gen_tokens, cfg.teacher_forced = P, N, False
cfg.tokens = P + N

t0 = time.perf_counter()
res = _core.run_generate(cfg, rows)                 # raises on a token error
dt = time.perf_counter() - t0

gen_ids = [int(t) for t in res.top1]
print(f"\ngenerated {res.completed} tokens in {dt:.0f}s ({dt / max(res.completed, 1):.0f}s/tok)")
print(f"bootstraps executed = {res.bootstraps}")

## The continuation

Decrypted only now. The encrypted run feeds back its own argmax each step, so any
divergence from the plaintext model compounds — matching for the first tokens and then
drifting is the expected behaviour, not a failure.

In [ ]:
print(f"prompt           : {tok.decode(ids)!r}")
print(f"FHE continuation : {tok.decode(gen_ids)!r}")
print(f"ref continuation : {tok.decode(ref)!r}")
agree = sum(a == b for a, b in zip(gen_ids, ref))
print(f"\ntoken agreement  : {agree}/{len(ref)}")
print(f"full FHE text    : {tok.decode(ids + gen_ids)!r}")
assert res.completed == N

## What this showed

- **Encrypted GPT-2** generated a continuation from a prompt of your choosing, with the
  sampled token never leaving the ciphertext domain (encrypted argmax feedback).
- It ran **eager**: every bootstrap was placed reactively by the runtime. The planned
  configuration of the paper is the oracle-fed decode row (`TASK=decode CHAIN=n32 bash
  scripts/run_task.sh`, 552 bootstraps per token against 1310 eager).
- Only `wte`+`wpe` ran client-side.

Change `PROMPT` in the first cell to prompt it differently — it must be at least
`GEN_PROMPT` tokens; `GEN_TOKENS` sets the length of the continuation.
`gpt2_perseus_nn.ipynb` does the same through the Python modules, with the client and the
server split across the trust boundary.